# Antara -- Fine-tune FILM on Colab

This notebook reuses the exact same pipeline code from the repo (`src/data`, `src/deep`, `src/eval`) so local dev and Colab stay in sync -- nothing here is a reimplementation, just orchestration.

**Before running**: `Runtime` -> `Change runtime type` -> `T4 GPU` (or better).

**Dataset design**: fine-tuning uses 3 separate GOES-16 days (2024-04-09, 2024-04-11, 2024-04-14), and evaluation uses a 4th day entirely (2024-04-19) that the model never sees during fine-tuning. This is stronger than a tail-slice of one contiguous run: a different day removes any chance of near-duplicate frames leaking between train and test, not just adjacent-triplet overlap within one run.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU -- set Runtime > Change runtime type > GPU before continuing")

## Clone the repo

The repo is private, so this needs a GitHub personal access token with `repo` scope (Settings -> Developer settings -> Personal access tokens). The token is only held in memory for this session, never written to disk or printed.

In [ ]:
from getpass import getpass

GITHUB_TOKEN = getpass("GitHub personal access token (repo scope): ")
REPO = "HimanshuSolo/Antara"

!git clone -q https://{GITHUB_TOKEN}@github.com/{REPO}.git antara
%cd antara

import sys
sys.path.insert(0, ".")
del GITHUB_TOKEN  # don't keep it around longer than needed

## Install dependencies

Skips `torch` on purpose -- Colab already has a CUDA-enabled build, and installing from `requirements.txt` as-is would pull the CPU-only wheel and silently lose GPU support.

In [ ]:
!pip install -q numpy opencv-python-headless scikit-image matplotlib xarray netCDF4 h5netcdf boto3 tqdm requests

## Configure data windows

Each training window is several hours from a distinct day. Adjust freely -- more/longer windows means more training data, at the cost of download time and disk (each scan is ~25MB).

In [ ]:
BAND = 13  # clean IR -- works day and night, ~25MB/scan vs 300MB+ for visible

TRAIN_WINDOWS = [
    ("2024-04-09T12:00", "2024-04-09T23:00"),
    ("2024-04-11T00:00", "2024-04-11T11:00"),
    ("2024-04-14T06:00", "2024-04-14T17:00"),
]

# A different day entirely, never touched during fine-tuning
TEST_WINDOW = ("2024-04-19T06:00", "2024-04-19T12:00")

## Download + extract triplets

In [ ]:
import shutil
from pathlib import Path

from src.data.fetch_goes import download_range
from src.data.extract_triplets import build_triplets, load_radiance
from datetime import datetime

def fetch_and_extract(start_str, end_str, raw_dir, triplets_dir):
    start = datetime.fromisoformat(start_str)
    end = datetime.fromisoformat(end_str)
    nc_paths = download_range(start, end, BAND, Path(raw_dir))

    sample_shape = load_radiance(nc_paths[0]).shape
    center = (sample_shape[0] // 2, sample_shape[1] // 2)

    written = build_triplets(nc_paths, center, size=256, out_dir=Path(triplets_dir))
    print(f"{triplets_dir}: {len(written)} triplets")
    return written

train_triplet_dirs = []
for i, (start, end) in enumerate(TRAIN_WINDOWS):
    written = fetch_and_extract(start, end, f"data/raw_train_{i}", f"data/processed/triplets_train_{i}")
    train_triplet_dirs.append(f"data/processed/triplets_train_{i}")

fetch_and_extract(*TEST_WINDOW, "data/raw_test", "data/processed/triplets_test")

## Merge the training windows into one fine-tuning pool

Each window's triplets are renumbered so they don't collide when combined into a single directory `TripletDataset` can load.

In [ ]:
merged_dir = Path("data/processed/triplets_finetune_all")
if merged_dir.exists():
    shutil.rmtree(merged_dir)
merged_dir.mkdir(parents=True)

idx = 0
for d in train_triplet_dirs:
    for triplet in sorted(Path(d).iterdir()):
        shutil.copytree(triplet, merged_dir / f"triplet_{idx:05d}")
        idx += 1

print(f"Total fine-tuning triplets: {idx}")

## Baseline numbers on the held-out test day (before any fine-tuning)

In [ ]:
from src.deep.download_film import download
from src.eval.evaluate_baseline import evaluate as evaluate_baseline
from src.eval.evaluate_film import evaluate as evaluate_film
from src.eval.metrics import summarize as _summarize_rows

pretrained_path = Path("models/film_net_fp32.pt")
download(
    "https://github.com/dajes/frame-interpolation-pytorch/releases/download/v1.0.2/film_net_fp32.pt",
    pretrained_path,
)

test_dir = Path("data/processed/triplets_test")

farneback_rows = evaluate_baseline(test_dir, Path("data/processed/test_farneback_results.csv"))
film_pretrained_rows = evaluate_film(test_dir, pretrained_path, Path("data/processed/test_film_pretrained_results.csv"))

def summarize(name, rows):
    psnr, ssim, lpips_dist = _summarize_rows(rows)
    print(f"{name:30s} PSNR={psnr:6.2f} dB   SSIM={ssim:.4f}   LPIPS={lpips_dist:.4f}")
    return psnr, ssim, lpips_dist

summarize("Farneback (classical)", farneback_rows)
summarize("FILM pretrained (zero-shot)", film_pretrained_rows)

## Fine-tune

GPU makes far more epochs/data feasible than the CPU proof-of-concept (5 epochs / 31 triplets) committed to the repo. Adjust `epochs`/`batch_size` based on how much time you want to spend -- a T4 should handle this comfortably.

In [ ]:
from src.deep.finetune_film import finetune

finetuned_path = Path("models/film_net_finetuned_colab.pt")

history = finetune(
    model_path=pretrained_path,
    triplets_dir=merged_dir,
    out_path=finetuned_path,
    epochs=30,
    batch_size=8,
    lr=1e-5,
    val_fraction=0.1,
)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("epoch")
plt.ylabel("L1 loss")
plt.legend()
plt.title("Fine-tuning loss")
plt.show()

## Final comparison on the held-out test day

In [ ]:
film_finetuned_rows = evaluate_film(test_dir, finetuned_path, Path("data/processed/test_film_finetuned_results.csv"))

print("=" * 60)
print("FINAL RESULT -- held-out test day, never seen during fine-tuning")
print("=" * 60)
summarize("Farneback (classical)", farneback_rows)
summarize("FILM pretrained (zero-shot)", film_pretrained_rows)
summarize("FILM fine-tuned", film_finetuned_rows)

## Save the fine-tuned checkpoint to Google Drive

Colab sessions are ephemeral -- without this, the checkpoint and results disappear when the runtime recycles.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

save_dir = Path("/content/drive/MyDrive/antara_finetune_results")
save_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(finetuned_path, save_dir / finetuned_path.name)
for csv_name in ["test_farneback_results.csv", "test_film_pretrained_results.csv", "test_film_finetuned_results.csv"]:
    shutil.copy(f"data/processed/{csv_name}", save_dir / csv_name)

print(f"Saved checkpoint + results to {save_dir}")